In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os
from datetime import datetime, timedelta

fake = Faker('en_IN')
random.seed(42)
np.random.seed(42)

print("All libraries loaded successfully")

In [ ]:
# Creating the folder structure in case it doesn't exist
os.makedirs('../data_raw', exist_ok=True)
os.makedirs('../data_cleaned', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

print("Folders confirmed ready")

In [ ]:
# CELL 19 — Fix duplicate branch names
import pandas as pd

df_branches = pd.read_csv('../data_raw/branches.csv')

# Add counter within each city to make names unique
df_branches['name_count'] = df_branches.groupby('branch_name').cumcount() + 1

# Only modify duplicates (count > 1)
mask = df_branches.duplicated(subset=['branch_name'], keep='first')
df_branches.loc[mask, 'branch_name'] = (
    df_branches.loc[mask, 'branch_name'] + ' ' + 
    df_branches.loc[mask, 'name_count'].astype(str)
)

df_branches.drop(columns=['name_count'], inplace=True)
df_branches.to_csv('../data_raw/branches.csv', index=False)

print(f"Duplicate names remaining: {df_branches['branch_name'].duplicated().sum()}")
print(df_branches[df_branches['city'] == 'Chennai'])

In [ ]:
# CELL 3 — Generate branch master list (120 branches)

cities_by_tier = {
    'Metro': ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Kolkata'],
    'Urban': ['Pune', 'Ahmedabad', 'Jaipur', 'Lucknow', 'Surat', 'Kanpur', 'Nagpur', 'Indore'],
    'Semi-Urban': ['Coimbatore', 'Vadodara', 'Bhopal', 'Patna', 'Ludhiana', 'Agra', 'Nashik', 'Ranchi'],
    'Rural': ['Warangal', 'Guntur', 'Mysuru', 'Jodhpur', 'Guwahati', 'Raipur', 'Dehradun', 'Tirupati']
}

tier_counts = {'Metro': 30, 'Urban': 35, 'Semi-Urban': 30, 'Rural': 25}

branches = []
branch_counter = 1

for tier, count in tier_counts.items():
    cities = cities_by_tier[tier]
    for i in range(count):
        city = cities[i % len(cities)]
        branch_id = f"BR{str(branch_counter).zfill(4)}"
        branches.append({
            'branch_id': branch_id,
            'branch_name': f"{city} {['Main', 'North', 'South', 'East', 'West', 'Central'][i % 6]} Branch",
            'city': city,
            'tier': tier,
            'region': 'South' if city in ['Hyderabad','Chennai','Bengaluru','Coimbatore','Warangal','Guntur','Mysuru','Tirupati'] else
                      'West'  if city in ['Mumbai','Pune','Ahmedabad','Surat','Vadodara','Nashik'] else
                      'North' if city in ['Delhi','Jaipur','Lucknow','Kanpur','Ludhiana','Agra','Jodhpur','Dehradun'] else 'East',
            'established_year': np.random.randint(1985, 2020)
        })
        branch_counter += 1

df_branches = pd.DataFrame(branches)
df_branches.to_csv('../data_raw/branches.csv', index=False)

print(f"Branches created: {len(df_branches)}")
print(df_branches['tier'].value_counts())
print(df_branches.head())

In [ ]:
# CELL 4 — Generate CBS (Core Banking System) monthly data

months = pd.date_range(start='2024-01-01', periods=12, freq='MS')

tier_base_transactions = {
    'Metro':      {'branch': 4500, 'digital': 8000},
    'Urban':      {'branch': 2800, 'digital': 4000},
    'Semi-Urban': {'branch': 1500, 'digital': 1200},
    'Rural':      {'branch':  800, 'digital':  300}
}

tier_base_financials = {
    'Metro':      {'deposits': 5000, 'fee_income': 45},
    'Urban':      {'deposits': 2200, 'fee_income': 22},
    'Semi-Urban': {'deposits':  900, 'fee_income':  9},
    'Rural':      {'deposits':  350, 'fee_income':  4}
}

cbs_rows = []

for _, branch in df_branches.iterrows():
    tier = branch['tier']
    base_txn = tier_base_transactions[tier]
    base_fin = tier_base_financials[tier]

    for month in months:
        seasonal = 1.0 + 0.1 * np.sin((month.month - 3) * np.pi / 6)

        branch_txns  = int(base_txn['branch']  * seasonal * np.random.uniform(0.85, 1.15))
        digital_txns = int(base_txn['digital'] * seasonal * np.random.uniform(0.85, 1.15))
        deposits     = round(base_fin['deposits']   * seasonal * np.random.uniform(0.88, 1.12), 1)
        fee_income   = round(base_fin['fee_income'] * np.random.uniform(0.80, 1.20), 2)

        cash_withdrawal_pct = np.random.uniform(0.45, 0.65)
        cash_deposit_pct    = np.random.uniform(0.15, 0.25)
        cheque_pct          = np.random.uniform(0.05, 0.12)
        other_pct           = 1 - cash_withdrawal_pct - cash_deposit_pct - cheque_pct

        cbs_rows.append({
            'branch_id':            branch['branch_id'],
            'month':                month.strftime('%Y-%m'),
            'branch_transactions':  branch_txns,
            'digital_transactions': digital_txns,
            'cash_withdrawals':     int(branch_txns * cash_withdrawal_pct),
            'cash_deposits':        int(branch_txns * cash_deposit_pct),
            'cheque_transactions':  int(branch_txns * cheque_pct),
            'other_transactions':   int(branch_txns * other_pct),
            'total_deposits_lakh':  deposits,
            'fee_income_lakh':      fee_income
        })

df_cbs = pd.DataFrame(cbs_rows)
df_cbs.to_csv('../data_raw/cbs_data.csv', index=False)

print(f"CBS rows generated: {len(df_cbs)}")
print(f"Months covered: {df_cbs['month'].nunique()}")
print(df_cbs.head(3))

In [ ]:
# CELL 5 — Generate HRMS (HR System) monthly data

# Staff count by tier
tier_staff = {
    'Metro':      {'tellers': 6, 'rms': 3, 'manager': 1},
    'Urban':      {'tellers': 4, 'rms': 2, 'manager': 1},
    'Semi-Urban': {'tellers': 3, 'rms': 1, 'manager': 1},
    'Rural':      {'tellers': 2, 'rms': 1, 'manager': 1}
}

# Monthly salary in rupees
salary = {
    'teller':  35000,
    'rm':      65000,
    'manager': 95000
}

# Working days per month (approx)
working_days_per_month = 26

hrms_rows = []

for _, branch in df_branches.iterrows():
    tier    = branch['tier']
    staff   = tier_staff[tier]
    n_tell  = staff['tellers']
    n_rm    = staff['rms']
    n_mgr   = staff['manager']

    for month in months:
        # Attendance rate — some branches have poor attendance (natural variation)
        teller_attendance  = np.random.uniform(0.78, 0.98)
        rm_attendance      = np.random.uniform(0.82, 0.98)
        manager_attendance = np.random.uniform(0.90, 1.00)

        # Actual days worked
        teller_days_worked  = round(n_tell * working_days_per_month * teller_attendance)
        rm_days_worked      = round(n_rm   * working_days_per_month * rm_attendance)
        manager_days_worked = round(n_mgr  * working_days_per_month * manager_attendance)

        # Total salary cost that month
        total_salary_cost = (
            n_tell * salary['teller']  +
            n_rm   * salary['rm']      +
            n_mgr  * salary['manager']
        )

        # Overhead cost: rent + electricity + misc (varies by tier)
        tier_overhead = {'Metro': 120000, 'Urban': 70000, 'Semi-Urban': 35000, 'Rural': 18000}
        overhead_cost = tier_overhead[tier] * np.random.uniform(0.90, 1.10)

        hrms_rows.append({
            'branch_id':           branch['branch_id'],
            'month':               month.strftime('%Y-%m'),
            'n_tellers':           n_tell,
            'n_rms':               n_rm,
            'n_manager':           n_mgr,
            'teller_attendance':   round(teller_attendance, 3),
            'rm_attendance':       round(rm_attendance, 3),
            'manager_attendance':  round(manager_attendance, 3),
            'teller_days_worked':  teller_days_worked,
            'rm_days_worked':      rm_days_worked,
            'total_salary_cost':   total_salary_cost,
            'overhead_cost':       round(overhead_cost, 0)
        })

df_hrms = pd.DataFrame(hrms_rows)
df_hrms.to_csv('../data_raw/hrms_data.csv', index=False)

print(f"HRMS rows generated: {len(df_hrms)}")
print(f"Total staff cost sample (BR0001, Jan): ₹{df_hrms[df_hrms['branch_id']=='BR0001'].iloc[0]['total_salary_cost']:,.0f}")
print(df_hrms.head(3))

In [ ]:
# CELL 6 — Generate CRM (Customer Relationship Management) monthly data

np.random.seed(99)  # fix zero-CRM assignment for consistency

crm_rows = []
for _, branch in df_branches.iterrows():
    tier = branch['tier']
    n_rm = {'Metro': 3, 'Urban': 2, 'Semi-Urban': 1, 'Rural': 1}[tier]
    branch_cbs = df_cbs[df_cbs['branch_id'] == branch['branch_id']]
    for _, cbs_row in branch_cbs.iterrows():
        footfall = cbs_row['branch_transactions']
        if np.random.rand() < 0.05:
            rm_interaction_rate = 0.0
        else:
            rm_interaction_rate = np.random.uniform(0.08, 0.35)
        interactions_logged  = int(footfall * rm_interaction_rate)
        followup_rate        = np.random.uniform(0.30, 0.80)
        followups_completed  = int(interactions_logged * followup_rate)
        conversion_rate      = np.random.uniform(0.05, 0.22)
        products_sold        = int(interactions_logged * conversion_rate)
        avg_loan_value = {'Metro': 35, 'Urban': 18, 'Semi-Urban': 8, 'Rural': 4}[tier]
        loan_value_disbursed = round(products_sold * avg_loan_value * np.random.uniform(0.8, 1.2), 1)
        crm_rows.append({
            'branch_id':                 branch['branch_id'],
            'month':                     cbs_row['month'],
            'footfall':                  footfall,
            'rm_interactions_logged':    interactions_logged,
            'rm_interaction_rate':       round(rm_interaction_rate, 3),
            'followups_completed':       followups_completed,
            'followup_rate':             round(followup_rate, 3),
            'products_sold':             products_sold,
            'conversion_rate':           round(conversion_rate, 3),
            'loan_value_disbursed_lakh': loan_value_disbursed,
            'is_zero_crm':               interactions_logged == 0
        })
df_crm = pd.DataFrame(crm_rows)
df_crm.to_csv('../data_raw/crm_data.csv', index=False)
print(f"CRM rows generated: {len(df_crm)}")
print(f"Zero-CRM months across all branches: {df_crm['is_zero_crm'].sum()}")
print(df_crm.head(3))

In [ ]:
# CELL 7 — Generate ATM (Cash Management) monthly data

np.random.seed(123)
months = pd.date_range(start='2024-01-01', periods=12, freq='MS')

# Not every branch has an ATM
# Metro: 100%, Urban: 85%, Semi-Urban: 60%, Rural: 40%
atm_probability = {'Metro': 1.0, 'Urban': 0.85, 'Semi-Urban': 0.60, 'Rural': 0.40}

# Cash loaded per replenishment cycle (in lakhs)
tier_cash_load = {'Metro': 25, 'Urban': 15, 'Semi-Urban': 8, 'Rural': 4}

# Replenishments per month
tier_replenishments = {'Metro': 8, 'Urban': 5, 'Semi-Urban': 3, 'Rural': 2}

atm_rows = []

for _, branch in df_branches.iterrows():
    tier = branch['tier']
    has_atm = np.random.rand() < atm_probability[tier]
    if not has_atm:
        continue
    for month in months:
        replenishments    = tier_replenishments[tier]
        cash_per_load     = tier_cash_load[tier]
        total_cash_loaded = round(replenishments * cash_per_load * np.random.uniform(0.90, 1.10), 1)
        utilisation_rate  = np.random.uniform(0.35, 0.90)
        cash_dispensed    = round(total_cash_loaded * utilisation_rate, 1)
        cash_idle         = round(total_cash_loaded - cash_dispensed, 1)
        downtime_hours    = round(np.random.exponential(scale=3.5), 1)
        atm_rows.append({
            'branch_id':               branch['branch_id'],
            'month':                   month.strftime('%Y-%m'),
            'has_atm':                 True,
            'replenishments':          replenishments,
            'total_cash_loaded_lakh':  total_cash_loaded,
            'cash_dispensed_lakh':     cash_dispensed,
            'cash_idle_lakh':          cash_idle,
            'utilisation_rate':        round(utilisation_rate, 3),
            'downtime_hours':          downtime_hours
        })

df_atm = pd.DataFrame(atm_rows)
df_atm.to_csv('../data_raw/atm_data.csv', index=False)

print(f"ATM rows generated: {len(df_atm)}")
print(f"Branches with ATM: {df_atm['branch_id'].nunique()} out of 120")
print(f"Avg utilisation rate: {df_atm['utilisation_rate'].mean():.1%}")
print(df_atm.head(3))

In [ ]:
# CELL 8 — Introduce realistic mess into raw data (simulating real-world data quality issues)

import random

# ── 1. CBS MESS ──────────────────────────────────────────────────────────────
df_cbs_messy = df_cbs.copy()

# a) Random missing values in fee_income_lakh (~4% of rows)
missing_idx = df_cbs_messy.sample(frac=0.04, random_state=1).index
df_cbs_messy.loc[missing_idx, 'fee_income_lakh'] = np.nan

# b) Random missing values in total_deposits_lakh (~2% of rows)
missing_idx2 = df_cbs_messy.sample(frac=0.02, random_state=2).index
df_cbs_messy.loc[missing_idx2, 'total_deposits_lakh'] = np.nan

# c) Inconsistent month format in ~5% of rows (some show as '01-2024' instead of '2024-01')
flip_idx = df_cbs_messy.sample(frac=0.05, random_state=3).index
df_cbs_messy.loc[flip_idx, 'month'] = df_cbs_messy.loc[flip_idx, 'month'].apply(
    lambda x: x[5:] + '-' + x[:4] if isinstance(x, str) else x
)

# d) Duplicate rows — 10 random rows duplicated
dupes = df_cbs_messy.sample(n=10, random_state=4)
df_cbs_messy = pd.concat([df_cbs_messy, dupes], ignore_index=True)

# e) Outlier: 3 branches have an impossibly high branch_transactions one month
outlier_idx = df_cbs_messy.sample(n=3, random_state=5).index
df_cbs_messy.loc[outlier_idx, 'branch_transactions'] = 999999


# ── 2. HRMS MESS ─────────────────────────────────────────────────────────────
df_hrms_messy = df_hrms.copy()

# a) Missing attendance values (~3% of rows)
missing_idx = df_hrms_messy.sample(frac=0.03, random_state=6).index
df_hrms_messy.loc[missing_idx, 'teller_attendance'] = np.nan

# b) Negative overhead cost in 5 rows (data entry error)
neg_idx = df_hrms_messy.sample(n=5, random_state=7).index
df_hrms_messy.loc[neg_idx, 'overhead_cost'] = -99999

# c) n_tellers = 0 in 4 rows (system export bug)
zero_idx = df_hrms_messy.sample(n=4, random_state=8).index
df_hrms_messy.loc[zero_idx, 'n_tellers'] = 0


# ── 3. CRM MESS ──────────────────────────────────────────────────────────────
df_crm_messy = df_crm.copy()

# a) Missing conversion_rate (~5% of rows)
missing_idx = df_crm_messy.sample(frac=0.05, random_state=9).index
df_crm_messy.loc[missing_idx, 'conversion_rate'] = np.nan

# b) rm_interactions_logged > footfall in 8 rows (impossible — more interactions than visitors)
impossible_idx = df_crm_messy.sample(n=8, random_state=10).index
df_crm_messy.loc[impossible_idx, 'rm_interactions_logged'] = \
    (df_crm_messy.loc[impossible_idx, 'footfall'] * np.random.uniform(1.2, 1.8)).astype(int)

# c) Duplicate rows — 8 random rows duplicated
dupes = df_crm_messy.sample(n=8, random_state=11)
df_crm_messy = pd.concat([df_crm_messy, dupes], ignore_index=True)


# ── 4. ATM MESS ──────────────────────────────────────────────────────────────
df_atm_messy = df_atm.copy()

# a) Missing utilisation_rate (~4% of rows)
missing_idx = df_atm_messy.sample(frac=0.04, random_state=12).index
df_atm_messy.loc[missing_idx, 'utilisation_rate'] = np.nan

# b) cash_dispensed > cash_loaded in 6 rows (physically impossible)
impossible_idx = df_atm_messy.sample(n=6, random_state=13).index
df_atm_messy.loc[impossible_idx, 'cash_dispensed_lakh'] = \
    df_atm_messy.loc[impossible_idx, 'total_cash_loaded_lakh'] * np.random.uniform(1.1, 1.5)

# c) Negative downtime in 4 rows (system error)
neg_idx = df_atm_messy.sample(n=4, random_state=14).index
df_atm_messy.loc[neg_idx, 'downtime_hours'] = -5.0


# ── SAVE ALL MESSY FILES ─────────────────────────────────────────────────────
df_cbs_messy.to_csv('../data_raw/cbs_messy.csv',   index=False)
df_hrms_messy.to_csv('../data_raw/hrms_messy.csv', index=False)
df_crm_messy.to_csv('../data_raw/crm_messy.csv',   index=False)
df_atm_messy.to_csv('../data_raw/atm_messy.csv',   index=False)

print("Mess introduced successfully")
print(f"CBS    — missing fee_income: {df_cbs_messy['fee_income_lakh'].isna().sum()} | dupes: 10 | outliers: 3 | bad month format: ~72")
print(f"HRMS   — missing attendance: {df_hrms_messy['teller_attendance'].isna().sum()} | negative overhead: 5 | zero tellers: 4")
print(f"CRM    — missing conversion: {df_crm_messy['conversion_rate'].isna().sum()} | impossible interactions: 8 | dupes: 8")
print(f"ATM    — missing utilisation: {df_atm_messy['utilisation_rate'].isna().sum()} | impossible dispense: 6 | negative downtime: 4")

In [ ]:
# CELL 9 — Data Audit (find all problems before fixing anything)

print("=" * 60)
print("CBS AUDIT")
print("=" * 60)
print(f"Total rows       : {len(df_cbs_messy)}")
print(f"Expected rows    : 1440 (120 branches x 12 months)")
print(f"Duplicate rows   : {df_cbs_messy.duplicated(subset=['branch_id','month']).sum()}")
print(f"Missing values:\n{df_cbs_messy.isnull().sum()}")
print(f"Unique month formats: {df_cbs_messy['month'].unique()[:10]}")
print(f"Max branch_transactions: {df_cbs_messy['branch_transactions'].max()}")
print(f"Min branch_transactions: {df_cbs_messy['branch_transactions'].min()}")

print("\n" + "=" * 60)
print("HRMS AUDIT")
print("=" * 60)
print(f"Total rows       : {len(df_hrms_messy)}")
print(f"Duplicate rows   : {df_hrms_messy.duplicated(subset=['branch_id','month']).sum()}")
print(f"Missing values:\n{df_hrms_messy.isnull().sum()}")
print(f"Negative overhead rows  : {(df_hrms_messy['overhead_cost'] < 0).sum()}")
print(f"Zero teller rows        : {(df_hrms_messy['n_tellers'] == 0).sum()}")
print(f"Min teller_attendance   : {df_hrms_messy['teller_attendance'].min()}")

print("\n" + "=" * 60)
print("CRM AUDIT")
print("=" * 60)
print(f"Total rows       : {len(df_crm_messy)}")
print(f"Duplicate rows   : {df_crm_messy.duplicated(subset=['branch_id','month']).sum()}")
print(f"Missing values:\n{df_crm_messy.isnull().sum()}")
impossible = (df_crm_messy['rm_interactions_logged'] > df_crm_messy['footfall']).sum()
print(f"Impossible interactions (logged > footfall): {impossible}")

print("\n" + "=" * 60)
print("ATM AUDIT")
print("=" * 60)
print(f"Total rows       : {len(df_atm_messy)}")
print(f"Duplicate rows   : {df_atm_messy.duplicated(subset=['branch_id','month']).sum()}")
print(f"Missing values:\n{df_atm_messy.isnull().sum()}")
impossible_atm = (df_atm_messy['cash_dispensed_lakh'] > df_atm_messy['total_cash_loaded_lakh']).sum()
print(f"Impossible dispense (dispensed > loaded): {impossible_atm}")
print(f"Negative downtime rows: {(df_atm_messy['downtime_hours'] < 0).sum()}")

In [ ]:
# CELL 10 — Clean CBS data

df_cbs_clean = df_cbs_messy.copy()

# ── FIX 1: Standardise month format ─────────────────────────────────────────
def fix_month_format(val):
    if isinstance(val, str):
        if len(val) == 7 and val[2] == '-':    # catches '05-2024' format
            return val[3:] + '-' + val[:2]
        return val
    return val

df_cbs_clean['month'] = df_cbs_clean['month'].apply(fix_month_format)
print(f"FIX 1 — Month format standardised")
print(f"Unique months now: {sorted(df_cbs_clean['month'].unique())}")


# ── FIX 2: Remove duplicates ─────────────────────────────────────────────────
before = len(df_cbs_clean)
df_cbs_clean = df_cbs_clean.drop_duplicates(subset=['branch_id', 'month'], keep='first')
print(f"\nFIX 2 — Duplicates removed: {before - len(df_cbs_clean)} rows dropped")


# ── FIX 3: Remove outliers in branch_transactions ────────────────────────────
# 999999 is a system overflow — drop these rows entirely
# Why drop and not fix? Because we have no way to know the real value
before = len(df_cbs_clean)
df_cbs_clean = df_cbs_clean[df_cbs_clean['branch_transactions'] < 50000]
print(f"\nFIX 3 — Outlier rows dropped: {before - len(df_cbs_clean)}")


# ── FIX 4: Handle missing total_deposits_lakh ────────────────────────────────
# Fill with the median deposit for that branch across other months
# Why median and not mean? Deposits can spike — median is more stable
df_cbs_clean['total_deposits_lakh'] = df_cbs_clean.groupby('branch_id')['total_deposits_lakh']\
    .transform(lambda x: x.fillna(x.median()))
print(f"\nFIX 4 — Missing deposits filled with branch median")
print(f"Remaining missing deposits: {df_cbs_clean['total_deposits_lakh'].isna().sum()}")


# ── FIX 5: Handle missing fee_income_lakh ────────────────────────────────────
# Fill with median fee income for that branch
df_cbs_clean['fee_income_lakh'] = df_cbs_clean.groupby('branch_id')['fee_income_lakh']\
    .transform(lambda x: x.fillna(x.median()))
print(f"\nFIX 5 — Missing fee income filled with branch median")
print(f"Remaining missing fee income: {df_cbs_clean['fee_income_lakh'].isna().sum()}")


# ── SAVE ─────────────────────────────────────────────────────────────────────
df_cbs_clean.to_csv('../data_cleaned/cbs_clean.csv', index=False)

print(f"\n── CBS CLEANING SUMMARY ──")
print(f"Rows before : 1450")
print(f"Rows after  : {len(df_cbs_clean)}")
print(f"Saved to    : data_cleaned/cbs_clean.csv")
print(df_cbs_clean.head(3))

In [ ]:
# CELL 11 — Clean HRMS data

df_hrms_clean = df_hrms_messy.copy()

# ── FIX 1: Handle missing teller_attendance ──────────────────────────────────
# Fill with median attendance for that branch
# Why not mean? A few very low months would pull mean down unfairly
df_hrms_clean['teller_attendance'] = df_hrms_clean.groupby('branch_id')['teller_attendance']\
    .transform(lambda x: x.fillna(x.median()))
print(f"FIX 1 — Missing teller attendance filled with branch median")
print(f"Remaining missing: {df_hrms_clean['teller_attendance'].isna().sum()}")


# ── FIX 2: Fix negative overhead costs ───────────────────────────────────────
# Negative overhead is physically impossible — replace with branch median
before = (df_hrms_clean['overhead_cost'] < 0).sum()
df_hrms_clean['overhead_cost'] = df_hrms_clean.groupby('branch_id')['overhead_cost']\
    .transform(lambda x: x.where(x >= 0, x[x >= 0].median()))
print(f"\nFIX 2 — Negative overhead rows fixed: {before}")
print(f"Remaining negative: {(df_hrms_clean['overhead_cost'] < 0).sum()}")


# ── FIX 3: Fix zero teller rows ───────────────────────────────────────────────
# n_tellers = 0 is a system bug — replace with the correct count for that branch
# Every branch row for same branch_id should have same n_tellers
before = (df_hrms_clean['n_tellers'] == 0).sum()
df_hrms_clean['n_tellers'] = df_hrms_clean.groupby('branch_id')['n_tellers']\
    .transform(lambda x: x.replace(0, x[x > 0].mode()[0]))
print(f"\nFIX 3 — Zero teller rows fixed: {before}")
print(f"Remaining zeros: {(df_hrms_clean['n_tellers'] == 0).sum()}")


# ── SAVE ─────────────────────────────────────────────────────────────────────
df_hrms_clean.to_csv('../data_cleaned/hrms_clean.csv', index=False)

print(f"\n── HRMS CLEANING SUMMARY ──")
print(f"Rows before : {len(df_hrms_messy)}")
print(f"Rows after  : {len(df_hrms_clean)}")
print(f"Saved to    : data_cleaned/hrms_clean.csv")
print(df_hrms_clean.head(3))

In [ ]:
# CELL 12 — Clean CRM data

df_crm_clean = df_crm_messy.copy()

# ── FIX 1: Remove duplicates ─────────────────────────────────────────────────
before = len(df_crm_clean)
df_crm_clean = df_crm_clean.drop_duplicates(subset=['branch_id', 'month'], keep='first')
print(f"FIX 1 — Duplicates removed: {before - len(df_crm_clean)} rows dropped")


# ── FIX 2: Fix impossible interactions (logged > footfall) ───────────────────
# rm_interactions cannot exceed the number of people who walked in
# Cap it at footfall value
impossible_mask = df_crm_clean['rm_interactions_logged'] > df_crm_clean['footfall']
df_crm_clean.loc[impossible_mask, 'rm_interactions_logged'] = \
    df_crm_clean.loc[impossible_mask, 'footfall']

# Recalculate rm_interaction_rate for these fixed rows
df_crm_clean['rm_interaction_rate'] = (
    df_crm_clean['rm_interactions_logged'] / df_crm_clean['footfall']
).round(3)

print(f"\nFIX 2 — Impossible interaction rows capped at footfall: {impossible_mask.sum()}")


# ── FIX 3: Handle missing conversion_rate ────────────────────────────────────
# Fill with median conversion rate for that branch
df_crm_clean['conversion_rate'] = df_crm_clean.groupby('branch_id')['conversion_rate']\
    .transform(lambda x: x.fillna(x.median()))
print(f"\nFIX 3 — Missing conversion rate filled with branch median")
print(f"Remaining missing: {df_crm_clean['conversion_rate'].isna().sum()}")


# ── FIX 4: Recalculate products_sold where conversion_rate was missing ────────
# For rows where products_sold is 0 but we now have a conversion rate,
# recalculate to keep data consistent
# (only for non-zero-crm rows)
mask = (df_crm_clean['products_sold'] == 0) & (df_crm_clean['is_zero_crm'] == False)
df_crm_clean.loc[mask, 'products_sold'] = (
    df_crm_clean.loc[mask, 'rm_interactions_logged'] *
    df_crm_clean.loc[mask, 'conversion_rate']
).astype(int)
print(f"\nFIX 4 — Products sold recalculated for {mask.sum()} rows")


# ── SAVE ─────────────────────────────────────────────────────────────────────
df_crm_clean.to_csv('../data_cleaned/crm_clean.csv', index=False)

print(f"\n── CRM CLEANING SUMMARY ──")
print(f"Rows before : {len(df_crm_messy)}")
print(f"Rows after  : {len(df_crm_clean)}")
print(f"Saved to    : data_cleaned/crm_clean.csv")
print(df_crm_clean.head(3))

In [ ]:
# CELL 13 — Clean ATM data

df_atm_clean = df_atm_messy.copy()

# ── FIX 1: Fix impossible dispense (cash_dispensed > cash_loaded) ────────────
# Physically impossible — ATM cannot dispense more than what was loaded
# Cap dispensed at loaded amount
impossible_mask = df_atm_clean['cash_dispensed_lakh'] > df_atm_clean['total_cash_loaded_lakh']
df_atm_clean.loc[impossible_mask, 'cash_dispensed_lakh'] = \
    df_atm_clean.loc[impossible_mask, 'total_cash_loaded_lakh']

# Recalculate cash_idle and utilisation_rate for fixed rows
df_atm_clean['cash_idle_lakh'] = (
    df_atm_clean['total_cash_loaded_lakh'] - df_atm_clean['cash_dispensed_lakh']
).round(1)
df_atm_clean['utilisation_rate'] = (
    df_atm_clean['cash_dispensed_lakh'] / df_atm_clean['total_cash_loaded_lakh']
).round(3)

print(f"FIX 1 — Impossible dispense rows fixed: {impossible_mask.sum()}")


# ── FIX 2: Fix negative downtime hours ───────────────────────────────────────
# Negative time is impossible — set to 0 (no downtime recorded)
before = (df_atm_clean['downtime_hours'] < 0).sum()
df_atm_clean['downtime_hours'] = df_atm_clean['downtime_hours'].clip(lower=0)
print(f"\nFIX 2 — Negative downtime rows fixed: {before}")
print(f"Remaining negative: {(df_atm_clean['downtime_hours'] < 0).sum()}")


# ── FIX 3: Handle missing utilisation_rate ───────────────────────────────────
# First try to recalculate from dispensed / loaded (most cases will be fine)
mask_missing = df_atm_clean['utilisation_rate'].isna()
df_atm_clean.loc[mask_missing, 'utilisation_rate'] = (
    df_atm_clean.loc[mask_missing, 'cash_dispensed_lakh'] /
    df_atm_clean.loc[mask_missing, 'total_cash_loaded_lakh']
).round(3)

# Any still missing — fill with branch median
df_atm_clean['utilisation_rate'] = df_atm_clean.groupby('branch_id')['utilisation_rate']\
    .transform(lambda x: x.fillna(x.median()))

print(f"\nFIX 3 — Missing utilisation rate fixed")
print(f"Remaining missing: {df_atm_clean['utilisation_rate'].isna().sum()}")


# ── SAVE ─────────────────────────────────────────────────────────────────────
df_atm_clean.to_csv('../data_cleaned/atm_clean.csv', index=False)

print(f"\n── ATM CLEANING SUMMARY ──")
print(f"Rows before : {len(df_atm_messy)}")
print(f"Rows after  : {len(df_atm_clean)}")
print(f"Saved to    : data_cleaned/atm_clean.csv")
print(df_atm_clean.head(3))

In [ ]:
# CELL 14 — Build Master Table (join all 4 systems + branches)

# ── STEP 1: Start with CBS as the base ───────────────────────────────────────
# CBS has 1437 rows — this is our spine for the master table
df_master = df_cbs_clean.copy()

# ── STEP 2: Merge branch info ─────────────────────────────────────────────────
df_master = df_master.merge(df_branches, on='branch_id', how='left')
print(f"After merging branches : {len(df_master)} rows, {df_master.shape[1]} columns")

# ── STEP 3: Merge HRMS ───────────────────────────────────────────────────────
df_master = df_master.merge(df_hrms_clean, on=['branch_id', 'month'], how='left')
print(f"After merging HRMS     : {len(df_master)} rows, {df_master.shape[1]} columns")

# ── STEP 4: Merge CRM ────────────────────────────────────────────────────────
df_master = df_master.merge(df_crm_clean, on=['branch_id', 'month'], how='left')
print(f"After merging CRM      : {len(df_master)} rows, {df_master.shape[1]} columns")

# ── STEP 5: Merge ATM (left join — not all branches have ATM) ────────────────
df_master = df_master.merge(
    df_atm_clean.drop(columns=['has_atm']),
    on=['branch_id', 'month'],
    how='left'
)
print(f"After merging ATM      : {len(df_master)} rows, {df_master.shape[1]} columns")

# ── STEP 6: Add ATM flag column ───────────────────────────────────────────────
df_master['has_atm'] = df_master['utilisation_rate'].notna()

# ── STEP 7: Check for any unexpected nulls after joining ─────────────────────
print(f"\n── NULL CHECK AFTER ALL JOINS ──")
null_counts = df_master.isnull().sum()
null_counts = null_counts[null_counts > 0]
print(null_counts)

print(f"\n── MASTER TABLE SHAPE ──")
print(f"Rows    : {len(df_master)}")
print(f"Columns : {df_master.shape[1]}")
print(f"\nAll columns:")
for col in df_master.columns:
    print(f"  {col}")

In [ ]:
# CELL 15 — Engineer the 4 core KPI metrics

# ── KPI 1: TELLER UTILISATION RATE ──────────────────────────────────────────
# What it measures: how busy tellers actually were vs their capacity
# Formula: branch_transactions / (n_tellers × teller_days_worked × transactions_per_day_capacity)
# A teller can handle ~25 transactions per day at full capacity

TELLER_DAILY_CAPACITY = 25

df_master['teller_capacity'] = (
    df_master['n_tellers'] *
    df_master['teller_days_worked'] *
    TELLER_DAILY_CAPACITY
)

df_master['teller_utilisation_rate'] = (
    df_master['branch_transactions'] / df_master['teller_capacity']
).round(3)

# Cap at 1.0 — cannot exceed 100% capacity
df_master['teller_utilisation_rate'] = df_master['teller_utilisation_rate'].clip(upper=1.0)

print(f"KPI 1 — Teller Utilisation Rate")
print(f"  Average : {df_master['teller_utilisation_rate'].mean():.1%}")
print(f"  Min     : {df_master['teller_utilisation_rate'].min():.1%}")
print(f"  Max     : {df_master['teller_utilisation_rate'].max():.1%}")


# ── KPI 2: COST PER TRANSACTION ─────────────────────────────────────────────
# What it measures: how much it costs the bank to process one branch transaction
# Formula: (salary + overhead) / branch_transactions
# Lower = more efficient

df_master['total_cost'] = df_master['total_salary_cost'] + df_master['overhead_cost']

df_master['cost_per_transaction'] = (
    df_master['total_cost'] / df_master['branch_transactions']
).round(2)

print(f"\nKPI 2 — Cost Per Transaction (₹)")
print(f"  Average : ₹{df_master['cost_per_transaction'].mean():.1f}")
print(f"  Min     : ₹{df_master['cost_per_transaction'].min():.1f}")
print(f"  Max     : ₹{df_master['cost_per_transaction'].max():.1f}")


# ── KPI 3: CASH IDLE RATIO ───────────────────────────────────────────────────
# What it measures: what % of loaded ATM cash sat unused
# Formula: cash_idle / cash_loaded
# Only for branches with ATM
# Higher = more waste

df_master['cash_idle_ratio'] = np.where(
    df_master['has_atm'],
    (df_master['cash_idle_lakh'] / df_master['total_cash_loaded_lakh']).round(3),
    np.nan
)

print(f"\nKPI 3 — Cash Idle Ratio (ATM branches only)")
print(f"  Average : {df_master['cash_idle_ratio'].mean():.1%}")
print(f"  Min     : {df_master['cash_idle_ratio'].min():.1%}")
print(f"  Max     : {df_master['cash_idle_ratio'].max():.1%}")


# ── KPI 4: RM ACTIVITY RATE ──────────────────────────────────────────────────
# What it measures: what % of footfall the RM actually engaged with
# Formula: rm_interactions_logged / footfall
# Already exists as rm_interaction_rate — we rename for clarity

df_master['rm_activity_rate'] = df_master['rm_interaction_rate']

print(f"\nKPI 4 — RM Activity Rate")
print(f"  Average : {df_master['rm_activity_rate'].mean():.1%}")
print(f"  Min     : {df_master['rm_activity_rate'].min():.1%}")
print(f"  Max     : {df_master['rm_activity_rate'].max():.1%}")


# ── SAVE MASTER TABLE ────────────────────────────────────────────────────────
df_master.to_csv('../data_cleaned/master_table.csv', index=False)

print(f"\n── MASTER TABLE SAVED ──")
print(f"Rows    : {len(df_master)}")
print(f"Columns : {df_master.shape[1]}")
print(f"Saved to: data_cleaned/master_table.csv")

In [ ]:
# CELL 16 — Branch Performance Analysis
# Goal: identify underperformers using the 4 KPIs
# Method: compare each branch against its tier benchmark
# Why tier? A Rural branch cannot be compared directly to a Metro branch

# ── STEP 1: Calculate tier benchmarks ────────────────────────────────────────
# For each tier, what is the median value of each KPI across all branches
# We use median not mean — outliers don't distort the benchmark

tier_benchmarks = df_master.groupby('tier').agg(
    benchmark_teller_utilisation  = ('teller_utilisation_rate', 'median'),
    benchmark_cost_per_transaction = ('cost_per_transaction',   'median'),
    benchmark_cash_idle_ratio      = ('cash_idle_ratio',        'median'),
    benchmark_rm_activity_rate     = ('rm_activity_rate',       'median')
).round(3)

print("── TIER BENCHMARKS ──")
print(tier_benchmarks)


# ── STEP 2: Calculate branch-level annual averages ───────────────────────────
# Collapse 12 months into one row per branch for comparison

branch_annual = df_master.groupby(['branch_id', 'branch_name', 'city', 'tier', 'region']).agg(
    avg_teller_utilisation   = ('teller_utilisation_rate', 'mean'),
    avg_cost_per_transaction = ('cost_per_transaction',    'mean'),
    avg_cash_idle_ratio      = ('cash_idle_ratio',         'mean'),
    avg_rm_activity_rate     = ('rm_activity_rate',        'mean'),
    total_deposits_lakh      = ('total_deposits_lakh',     'sum'),
    total_fee_income_lakh    = ('fee_income_lakh',         'sum'),
    total_products_sold      = ('products_sold',           'sum'),
    zero_crm_months          = ('is_zero_crm',             'sum'),
    avg_downtime_hours       = ('downtime_hours',          'mean'),
    has_atm                  = ('has_atm',                 'first')
).round(3).reset_index()

print(f"\nBranch annual summary: {len(branch_annual)} branches")


# ── STEP 3: Merge tier benchmarks into branch annual ─────────────────────────
branch_annual = branch_annual.merge(tier_benchmarks, on='tier', how='left')


# ── STEP 4: Flag underperformers on each KPI ─────────────────────────────────
# A branch is underperforming on a KPI if it is worse than its tier benchmark

# Teller utilisation — BELOW benchmark is bad (tellers are idle)
branch_annual['flag_teller'] = (
    branch_annual['avg_teller_utilisation'] < branch_annual['benchmark_teller_utilisation']
).astype(int)

# Cost per transaction — ABOVE benchmark is bad (costs too much)
branch_annual['flag_cost'] = (
    branch_annual['avg_cost_per_transaction'] > branch_annual['benchmark_cost_per_transaction']
).astype(int)

# Cash idle ratio — ABOVE benchmark is bad (too much idle cash)
branch_annual['flag_cash'] = (
    branch_annual['avg_cash_idle_ratio'] > branch_annual['benchmark_cash_idle_ratio']
).astype(int)

# RM activity rate — BELOW benchmark is bad (RM not engaging customers)
branch_annual['flag_rm'] = (
    branch_annual['avg_rm_activity_rate'] < branch_annual['benchmark_rm_activity_rate']
).astype(int)


# ── STEP 5: Overall underperformance score ────────────────────────────────────
# Sum of flags — max 4, min 0
# 3 or 4 flags = serious underperformer
# 2 flags = watch list
# 0-1 flags = healthy

branch_annual['underperformance_score'] = (
    branch_annual['flag_teller'] +
    branch_annual['flag_cost']   +
    branch_annual['flag_cash']   +
    branch_annual['flag_rm']
)

# ── STEP 6: Assign performance label ─────────────────────────────────────────
def performance_label(score):
    if score >= 3:   return 'Critical'
    elif score == 2: return 'Watch'
    else:            return 'Healthy'

branch_annual['performance'] = branch_annual['underperformance_score'].apply(performance_label)


# ── STEP 7: Print key findings ────────────────────────────────────────────────
print(f"\n── PERFORMANCE DISTRIBUTION ──")
print(branch_annual['performance'].value_counts())

print(f"\n── TOP 10 WORST BRANCHES ──")
worst = branch_annual.sort_values('underperformance_score', ascending=False).head(10)
print(worst[['branch_id','branch_name','tier','underperformance_score','performance',
             'avg_teller_utilisation','avg_cost_per_transaction',
             'avg_rm_activity_rate','zero_crm_months']].to_string())

print(f"\n── ZERO CRM BRANCHES (RM logged nothing for 1+ months) ──")
zero_crm = branch_annual[branch_annual['zero_crm_months'] > 0].sort_values('zero_crm_months', ascending=False)
print(f"Branches with at least 1 zero-CRM month: {len(zero_crm)}")
print(zero_crm[['branch_id','branch_name','tier','zero_crm_months']].head(10).to_string())

# ── SAVE ─────────────────────────────────────────────────────────────────────
branch_annual.to_csv('../outputs/branch_annual_performance.csv', index=False)
print(f"\nSaved to: outputs/branch_annual_performance.csv")

In [ ]:
# CELL 17 — Deep Dive Analysis
# 3 angles: Region, Tier, and Deposit vs Efficiency trap

# ── ANALYSIS 1: Performance by Region ────────────────────────────────────────
print("── PERFORMANCE BY REGION ──")
region_analysis = branch_annual.groupby('region').agg(
    total_branches        = ('branch_id',                'count'),
    critical_branches     = ('performance',              lambda x: (x == 'Immediate Action').sum()),
    avg_teller_util       = ('avg_teller_utilisation',   'mean'),
    avg_cost_per_txn      = ('avg_cost_per_transaction', 'mean'),
    avg_rm_activity       = ('avg_rm_activity_rate',     'mean'),
    avg_cash_idle         = ('avg_cash_idle_ratio',      'mean'),
    total_deposits        = ('total_deposits_lakh',      'sum'),
    total_products_sold   = ('total_products_sold',      'sum')
).round(3)

region_analysis['critical_pct'] = (
    region_analysis['critical_branches'] / region_analysis['total_branches'] * 100
).round(1)

print(region_analysis[[
    'total_branches','critical_branches','critical_pct',
    'avg_teller_util','avg_cost_per_txn','avg_rm_activity'
]].to_string())


# ── ANALYSIS 2: Performance by Tier ──────────────────────────────────────────
print("\n── PERFORMANCE BY TIER ──")
tier_analysis = branch_annual.groupby('tier').agg(
    total_branches      = ('branch_id',                'count'),
    critical_branches   = ('performance',              lambda x: (x == 'Immediate Action').sum()),
    avg_teller_util     = ('avg_teller_utilisation',   'mean'),
    avg_cost_per_txn    = ('avg_cost_per_transaction', 'mean'),
    avg_rm_activity     = ('avg_rm_activity_rate',     'mean'),
    total_deposits      = ('total_deposits_lakh',      'sum'),
    total_products_sold = ('total_products_sold',      'sum')
).round(3)

tier_analysis['critical_pct'] = (
    tier_analysis['critical_branches'] / tier_analysis['total_branches'] * 100
).round(1)

print(tier_analysis[[
    'total_branches','critical_branches','critical_pct',
    'avg_teller_util','avg_cost_per_txn','avg_rm_activity'
]].to_string())


# ── ANALYSIS 3: The Deposit Trap ─────────────────────────────────────────────
print("\n── THE DEPOSIT TRAP (high deposits + Immediate Action performance) ──")
deposit_median = branch_annual['total_deposits_lakh'].median()

deposit_trap = branch_annual[
    (branch_annual['performance'] == 'Immediate Action') &
    (branch_annual['total_deposits_lakh'] > deposit_median)
].sort_values('total_deposits_lakh', ascending=False)

print(f"Branches that look profitable but are operationally critical: {len(deposit_trap)}")
print(deposit_trap[[
    'branch_id','branch_name','tier',
    'total_deposits_lakh','avg_cost_per_transaction',
    'avg_teller_utilisation','avg_rm_activity_rate','zero_crm_months'
]].to_string())


# ── ANALYSIS 4: Monthly trend ────────────────────────────────────────────────
print("\n── MONTHLY TREND (all branches avg KPIs) ──")
monthly_trend = df_master.groupby('month').agg(
    avg_teller_util  = ('teller_utilisation_rate', 'mean'),
    avg_cost_per_txn = ('cost_per_transaction',    'mean'),
    avg_rm_activity  = ('rm_activity_rate',        'mean'),
    avg_cash_idle    = ('cash_idle_ratio',         'mean')
).round(3)

print(monthly_trend.to_string())

# ── SAVE ─────────────────────────────────────────────────────────────────────
region_analysis.to_csv('../outputs/region_analysis.csv')
tier_analysis.to_csv('../outputs/tier_analysis.csv')
deposit_trap.to_csv('../outputs/deposit_trap_branches.csv', index=False)
monthly_trend.to_csv('../outputs/monthly_trend.csv')

print(f"\nAll analysis files saved to outputs/")

In [ ]:
import os
print(os.path.abspath('../outputs'))

In [ ]:
# CELL 18 — Relabel performance categories

def relabel(score):
    if score >= 3:   return 'Immediate Action'
    elif score == 2: return 'Monitor'
    else:            return 'Healthy'

branch_annual['performance'] = branch_annual['underperformance_score'].apply(relabel)

# Resave
branch_annual.to_csv('../outputs/branch_annual_performance.csv', index=False)

print(branch_annual['performance'].value_counts())

In [ ]:
import pandas as pd
df = pd.read_csv('../outputs/monthly_trend.csv')
print(df.dtypes)
print(df.head())

In [ ]:
# CELL 19 — Fix month labels for Power BI display
import pandas as pd

df = pd.read_csv('../outputs/monthly_trend.csv')

# Convert to short month name
df['month'] = pd.to_datetime(df['month']).dt.strftime('%b')

df.to_csv('../outputs/monthly_trend.csv', index=False)
print(df)

In [ ]:
# CELL 19 fixed — proper month labels
import pandas as pd

df = pd.read_csv('../outputs/monthly_trend.csv')

# Fix month column back to proper format
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df['month'] = months

df.to_csv('../outputs/monthly_trend.csv', index=False)
print(df[['month']].to_string())

In [ ]:
# Revert monthly_trend month column back to original
import pandas as pd

months = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06',
          '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']

df = pd.read_csv('../outputs/monthly_trend.csv')
df['month'] = months
df.to_csv('../outputs/monthly_trend.csv', index=False)
print(df[['month']])

In [ ]:
print(branch_annual.groupby('region')['avg_rm_activity_rate'].mean().round(3))

In [ ]:
print(branch_annual.groupby('tier')['avg_cash_idle_ratio'].mean().round(3))

In [ ]:
print(branch_annual[branch_annual['branch_name'] == 'Chennai West Branch'])